In [ ]:
!pip install transformers
!pip install torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

In [ ]:
# Function to generate chatbot response
def generate_response(user_input, chat_history_ids):
    # Encode user input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append to chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=100,
        top_p=0.7,
        temperature=0.8
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [ ]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("User: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!,always read to assist you")
        break

    # Hardcoded responses (exact flow)
    if user_input.lower() == "hello":
        print("Chatbot: Hello! Nice to meet you. How can I assist you today?")
        continue

    if user_input.lower() == "what is artificial intelligence?":
        print("Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.")
        continue

    if user_input.lower() == "who created python?":
        print("Chatbot: Python was created by Guido van Rossum and released in 1991.")
        continue

    if user_input.lower() == "thank you":
        print("Chatbot: You're welcome! Feel free to ask more questions.")
        continue
    # Add to improve responses
    formatted_input = "User: " + user_input + " Bot:"

    new_input_ids = tokenizer.encode(formatted_input + tokenizer.eos_token, return_tensors='pt')

        # Attention mask
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        # Generate response
    with torch.no_grad():
            chat_history_ids = model.generate(
                bot_input_ids,
                attention_mask=attention_mask,
                max_length=200,
                pad_token_id=tokenizer.eos_token_id,
                no_repeat_ngram_size=3,
                do_sample=True,
                top_k=50,
                top_p=0.9,
                temperature=0.6
            )

        # Decode response
    response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

    print("🤖 Chatbot:", response, "\n")